# App-10b : Optimisation de portefeuille par algorithme genetique (C#)

**Navigation** : [Index](../../README.md) | [<< App-10 Python](../Hybrid/App-10-Portfolio.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Modeliser** un portefeuille financier (rendement, risque, matrice de covariance)
2. **Appliquer** un algorithme genetique (GeneticSharp) a l'optimisation de portefeuille
3. **Implémenter** une fonction fitness avec aversion au risque (parametre alpha)
4. **Configurer** les operateurs genetiques (selection, crossover, mutation) pour la finance

### Prerequis
- .NET 9.0 Interactive (C#)
- Search-5 : Algorithmes genetiques (concepts de fitness, selection, crossover, mutation)
- Notions de base en finance (rendement, risque, diversification)

### Duree estimee : 30 minutes

> **Side track** : Voir [App-10 Python](../Hybrid/App-10-Portfolio.ipynb) pour la version PyGAD de ce probleme avec frontiere efficiente de Markowitz.

**Navigation** : [Index](../../../README.md) | [<< App-10 Python](App-10-Portfolio.ipynb)

## Installation de GeneticSharp


In [1]:
#r "nuget: GeneticSharp"

Installed Packages GeneticSharp, 3.1.4

## Définition et création du modèle de portefeuille

Nous modélisons ici un portefeuille d'actifs. Chaque actif est muni d'un prix, d'un rendement attendu, et participe à une matrice de covariance globale prise en compte dans la fonction de risque.  

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

// Classe Portfolio pour représenter un portefeuille d'actifs financiers
public class Portfolio
{
    // Liste des noms d'actifs
    public List<string> Assets { get; set; }

    // Dictionnaire avec les prix des actifs
    public Dictionary<string, double> Prices { get; set; }

    // Dictionnaire contenant les rendements attendus des actifs
    public Dictionary<string, double> ExpectedReturns { get; set; }

    // Matrice de covariance (taille NxN pour N actifs). 
    // CovarianceMatrix[i,j] représente la covariance entre l'actif i et j.
    public double[,] CovarianceMatrix { get; set; }

    // Constructeur pour initialiser le portefeuille
    public Portfolio(List<string> assets,
                     Dictionary<string, double> prices,
                     Dictionary<string, double> expectedReturns,
                     double[,] covarianceMatrix)
    {
        Assets = assets;
        Prices = prices;
        ExpectedReturns = expectedReturns;
        CovarianceMatrix = covarianceMatrix;
    }

    // Calcul du retour total du portefeuille
    public double CalculateReturn(Dictionary<string, double> weights)
    {
        double portfolioReturn = 0;
        foreach (var asset in Assets)
        {
            portfolioReturn += weights[asset] * ExpectedReturns[asset];
        }
        return portfolioReturn;
    }

    // Calcul du risque (écart-type) du portefeuille en utilisant la matrice de covariance
    public double CalculateRisk(Dictionary<string, double> weights)
    {
        // On convertit le dictionnaire de poids en un vecteur aligné sur l'ordre de la liste Assets
        var weightVector = new double[Assets.Count];
        for(int i = 0; i < Assets.Count; i++)
        {
            weightVector[i] = weights[Assets[i]];
        }

        // Calcul de la variance via wᵀΣw
        double variance = 0.0;
        int n = Assets.Count;
        for(int i = 0; i < n; i++)
        {
            for(int j = 0; j < n; j++)
            {
                variance += weightVector[i] * weightVector[j] * CovarianceMatrix[i, j];
            }
        }

        // Le risque est la racine carrée de la variance
        return Math.Sqrt(variance);
    }
}

Console.WriteLine("Classes et interfaces definies.");


Classes et interfaces definies.


## Définition du chromosome pour l'algorithme génétique

Chaque chromosome représente une allocation de portefeuille. Chaque gène correspond au poids d'un actif, et la somme des poids est toujours normalisée à 1.

In [3]:
using GeneticSharp;
using System.Linq;

// Classe pour représenter un chromosome de portefeuille
public class PortfolioChromosome : ChromosomeBase
{
    private List<string> _assets;

    // Constructeur : on crée un chromosome avec un gène par actif
    public PortfolioChromosome(List<string> assets) : base(assets.Count)
    {
        _assets = assets;
        for (int i = 0; i < assets.Count; i++)
        {
            ReplaceGene(i, new Gene(RandomizationProvider.Current.GetDouble(0, 1)));
        }
    }

    // Génère un gène avec une valeur aléatoire entre 0 et 1
    public override Gene GenerateGene(int geneIndex)
    {
        return new Gene(RandomizationProvider.Current.GetDouble(0, 1));
    }

    // Crée un nouveau chromosome
    public override IChromosome CreateNew()
    {
        return new PortfolioChromosome(_assets);
    }

    // Récupère la distribution des poids, normalisés pour que la somme = 1
    public Dictionary<string, double> GetWeights()
    {
        var weights = new Dictionary<string, double>();
        for (int i = 0; i < Length; i++)
        {
            weights.Add(_assets[i], (double)GetGene(i).Value);
        }
        NormalizeWeights(weights);
        return weights;
    }

    // Normalise les poids pour que leur somme soit égale à 1
    private void NormalizeWeights(Dictionary<string, double> weights)
    {
        double sum = weights.Values.Sum();
        var keys = weights.Keys.ToList();
        foreach (var key in keys)
        {
            weights[key] /= sum;
        }
    }
}

Console.WriteLine("PortfolioChromosome et helpers definis.");


PortfolioChromosome et helpers definis.


## Implémentation du Solveur par Algorithme Génétique

L'implémentation du solveur par algorithme génétique se fait en plusieurs étapes, incluant la définition des classes nécessaires, la configuration de l'algorithme génétique et l'exécution de l'algorithme pour trouver la meilleure solution possible.

In [4]:
// Classe pour évaluer la fitness d'un portefeuille
public class PortfolioFitness : IFitness
{
    private readonly Portfolio _portfolio;
    private readonly double _alpha;

    // Constructeur: le paramètre alpha permet de moduler l'aversion au risque
    public PortfolioFitness(Portfolio portfolio, double alpha = 1.0)
    {
        _portfolio = portfolio;
        _alpha = alpha;
    }

    // Fonction d'évaluation de la fitness
    public double Evaluate(IChromosome chromosome)
    {
        var portfolioChromosome = chromosome as PortfolioChromosome;
        var weights = portfolioChromosome.GetWeights();

        double portfolioReturn = _portfolio.CalculateReturn(weights);
        double risk = _portfolio.CalculateRisk(weights);

        // Objectif : maximiser le retour, minimiser le risque.
        // Fitness = retour - alpha*risk
        double fitness = portfolioReturn - _alpha * risk;
        return fitness;
    }
}

Console.WriteLine("PortfolioFitness defini.");


PortfolioFitness defini.


## Exemple d'utilisation et exécution de l'algorithme génétique

Dans cet exemple, nous générons cinq actifs différents avec des rendements attendus croissants.  
Une matrice de covariance simple leur est associée.  
Nous configurerons ensuite l'algorithme génétique avec une sélection par roulette, un crossover uniforme et une légère mutation.  
On ajoutera également un peu d'élitisme et un nombre de générations supérieur afin de réellement progresser dans la recherche d'une solution optimale.  

In [5]:
using GeneticSharp;

// Liste d'actifs
var assets = new List<string> { "Asset0", "Asset1", "Asset2", "Asset3", "Asset4" };

// Prix (exemple de dummy data)
var prices = new Dictionary<string, double>
{
    { "Asset0", 100 },
    { "Asset1", 200 },
    { "Asset2", 300 },
    { "Asset3", 400 },
    { "Asset4", 500 }
};

// Rendements attendus
var expectedReturns = new Dictionary<string, double>
{
    { "Asset0", 0.05 },
    { "Asset1", 0.10 },
    { "Asset2", 0.15 },
    { "Asset3", 0.20 },
    { "Asset4", 0.25 }
};

// Matrice de covariance (5x5) hypothétique
double[,] covarianceMatrix = new double[,]
{
    { 0.0100, 0.0012, 0.0018, 0.0021, 0.0025 },
    { 0.0012, 0.0200, 0.0022, 0.0026, 0.0028 },
    { 0.0018, 0.0022, 0.0300, 0.0031, 0.0033 },
    { 0.0021, 0.0026, 0.0031, 0.0400, 0.0043 },
    { 0.0025, 0.0028, 0.0033, 0.0043, 0.0500 }
};

// Instanciation du portefeuille
var portfolio = new Portfolio(assets, prices, expectedReturns, covarianceMatrix);

// Paramètre alpha = 1.0 (vous pouvez l'ajuster selon votre tolérance au risque)
var fitness = new PortfolioFitness(portfolio, alpha: 1.0);

// On crée un chromosome représentatif
var chromosome = new PortfolioChromosome(assets);

// Population initiale de 50 solutions, jusqu'à 100 solutions, sur la base du chromosome ci-dessus
var population = new Population(50, 100, chromosome);

// Configuration de l'algorithme génétique
var ga = new GeneticAlgorithm(
    population,
    fitness,
    new RouletteWheelSelection(),
    new UniformCrossover(),
    new UniformMutation()
);

// Nombre maximum de générations
ga.Termination = new GenerationNumberTermination(150);

// On peut configurer un peu d'élitisme et ajuster les probabilités
ga.Selection = new EliteSelection();
ga.MutationProbability = 0.05f;
ga.CrossoverProbability = 0.8f;

// Démarrage de l'algorithme
ga.Start();

// Récupération du meilleur chromosome
var bestChromosome = ga.BestChromosome as PortfolioChromosome;
var bestWeights = bestChromosome.GetWeights();

// Affichage des allocations
Console.WriteLine("=== Meilleure allocation d'actifs trouvée ===");
foreach (var asset in bestWeights.Keys)
{
    Console.WriteLine($"{asset,-10}: {bestWeights[asset]:P2}");
}

// Affichage du rendement et du risque
double bestReturn = portfolio.CalculateReturn(bestWeights);
double bestRisk = portfolio.CalculateRisk(bestWeights);
Console.WriteLine($"\nRendement attendu : {bestReturn:P2}");
Console.WriteLine($"Risque (écart-type): {bestRisk:P2}");
Console.WriteLine($"Fitness (return - alpha*risk): {bestReturn - bestRisk:0.0000}");

=== Meilleure allocation d'actifs trouvée ===


Asset0    : 4,31 %


Asset1    : 5,29 %


Asset2    : 22,32 %


Asset3    : 31,12 %


Asset4    : 36,96 %



Rendement attendu : 19,56 %


Risque (écart-type): 12,11 %


Fitness (return - alpha*risk): 0,0745


### Interpretation : Resultats de l'algorithme genetique

**Sortie obtenue** : L'algorithme genetique a converge vers une allocation de portefeuille avec un rendement attendu de 19,56% et un risque de 12,11% (fitness 0,0745).

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Asset0 | 4,31% | Allocation faible |
| Asset1 | 5,29% | Allocation faible |
| Asset2 | 22,32% | Allocation significative |
| Asset3 | 31,12% | Poids important |
| Asset4 | 36,96% | Allocation maximale |
| Rendement attendu | 19,56% | Rendement pondere des 5 actifs |
| Risque (ecart-type) | 12,11% | Volatilite du portefeuille |
| Fitness | 0,0745 | Rendement - alpha * risque (alpha=1.0) |

**Points cles** :
1. L'algorithme a naturellement favorise les actifs a haut rendement (Asset3 et Asset4) tout en controlant le risque via la matrice de covariance
2. La penalisation du risque (parametre alpha=1.0) evite une concentration excessive sur l'actif le plus rentable
3. La fitness finale de 0,0745 represente le compromis optimal entre rendement et risque pour ce parametre alpha

> **Note technique** : La matrice de covariance joue un role crucial dans la fonction de risque. Les correlations positives entre actifs signifient que la diversification reduit le risque mais pas autant que si les actifs etaient parfaitement decores.

## Exercices

### Exercice 1 : Contraintes de Poids Minimaux et Maximaux

En pratique, un gestionnaire de portefeuille impose souvent des limites sur la proportion de chaque actif. Modifiez la classe `PortfolioChromosome` pour respecter ces contraintes.

**Indices** :
- Ajoutez des dictionnaires `min_weights` et `max_weights` en parametres
- Apres normalisation, verifiez que chaque poids respecte les contraintes
- Si une contrainte est violee, ajustez les poids de maniere proportionnelle

In [6]:
// TODO: Implementer une version contrainte de PortfolioChromosome
public class ConstrainedPortfolioChromosome : ChromosomeBase
{
    private List<string> _assets;
    private Dictionary<string, double> _minWeights;
    private Dictionary<string, double> _maxWeights;

    public ConstrainedPortfolioChromosome(
        List<string> assets,
        Dictionary<string, double> minWeights,
        Dictionary<string, double> maxWeights) : base(assets.Count)
    {
        // TODO etudiant : initialiser les champs et les genes
        Console.WriteLine("Exercice a completer");
    }

    public override Gene GenerateGene(int geneIndex)
    {
        // TODO etudiant : retourner un gene avec valeur aleatoire entre 0 et 1
        return new Gene(0.0); // TODO etudiant : utiliser RandomizationProvider
    }

    public override IChromosome CreateNew()
    {
        // TODO etudiant : retourner une nouvelle instance de ConstrainedPortfolioChromosome
        return null; // TODO etudiant
    }

    public Dictionary<string, double> GetWeights()
    {
        // TODO etudiant : recuperer les genes, normaliser, et appliquer les contraintes
        return null; // TODO etudiant
    }

    // Indice: Implementer une methode pour ajuster les poids aux contraintes
    private void ApplyConstraints(Dictionary<string, double> weights)
    {
        // TODO etudiant : ajuster les poids pour respecter min_weights et max_weights
    }
}


(6,40): warning CS0169: Le champ 'ConstrainedPortfolioChromosome._maxWeights' n'est jamais utilisé

(5,40): warning CS0169: Le champ 'ConstrainedPortfolioChromosome._minWeights' n'est jamais utilisé

(4,26): warning CS0169: Le champ 'ConstrainedPortfolioChromosome._assets' n'est jamais utilisé



### Exercice 2 : Selection par Tournoi

Implementez une selection par tournoi pour comparer ses performances avec la selection Elite utilisee dans l'exemple.

**Indices** :
- Dans un tournoi, k individus sont selectionnes aleatoirement
- Le meilleur des k est choisi comme parent
- Comparez la vitesse de convergence et la qualite finale avec EliteSelection

In [7]:
// TODO: Comparer EliteSelection et TournamentSelection
// Indice: GeneticSharp fournit TournamentSelection, il suffit de configurer l'algorithme

public void CompareSelectionMethods(Portfolio portfolio, List<string> assets, int generations = 100)
{
    // A completer
    Console.WriteLine("Exercice a completer");
    
    // TODO: Executer l'AG avec EliteSelection et mesurer la fitness finale
    // A completer
    
    // TODO: Executer l'AG avec TournamentSelection (taille tournoi = 3)
    // A completer
    
    // TODO: Afficher les resultats comparatifs
    // A completer
}

### Exercice 3 : Frontiere Efficiente

Tracez la **frontiere efficiente** de Markowitz en faisant varier le parametre alpha systematiquement.

**Indices** :
- Faites varier alpha de 0.1 a 3.0 par pas de 0.3
- Pour chaque valeur, executez l'AG et collectez le rendement et le risque
- Affichez les points (risque, rendement) pour former la frontiere

**Questions** : Comment evolue le ratio rendement/risque quand alpha augmente ? Quel alpha offre le meilleur compromis ?

In [8]:
// --- Exercice 3 : Frontiere Efficiente ---

// TODO 1: Definir les valeurs de alpha a tester
// var alphaValues = new[] { 0.1, 0.5, 1.0, 1.5, 2.0, 3.0 };

// TODO 2: Pour chaque alpha, executer l'AG et collecter (rendement, risque)
// foreach (var alpha in alphaValues) {
//     var fitness = new PortfolioFitness(portfolio, alpha);
//     // ... configurer et executer l'AG ...
//     Console.WriteLine($"alpha={alpha}: Return={ret:P2}, Risk={risk:P2}");
// }

// TODO 3: Identifier le meilleur compromis rendement/risque

Console.WriteLine("Exercice a completer");

Exercice a completer


## Conclusion

Avec cet exemple plus élaboré, nous prenons mieux en compte la structure de corrélation/risque entre actifs grâce à la matrice de covariance, tout en modulant l'aversion au risque avec un paramètre alpha.  
L'algorithme génétique est configuré pour un plus grand nombre de générations, de l'élitisme, et des probabilités de croisement et de mutation afin d'accroître ses chances de trouver une allocation de portefeuille optimale.  

Pour aller encore plus loin, vous pourriez :  
- Expérimenter avec différentes méthodes de sélection (RouletteWheel, Tournament, Elite, etc.).  
- Ajuster la valeur de alpha pour augmenter ou diminuer la pénalisation du risque.  
- Introduire des contraintes supplémentaires (ex. poids minimum/maximum par actif).  
- Logger l'évolution de la fitness dans le temps pour observer la convergence de l'algorithme.

**Retour au sommaire** : [Index](../../README.md)